# 05 — Fetch Subscriptions

Pulls subscriptions from MySQL per `SUBSCRIPTION_QUERY` (currently scoped to
a single `AccountCode` for testing — see the query in step 2), then for each
one:

1. Decides which target account it belongs to (`resolve_target_accounts`,
   based on `Reference` — see step 3 below for the full priority order).
2. Looks up its real address + radius username from Voyager
   (`get_voyager_address`) — `SupplierServiceID` -> `GET .../fibre/v1/circuits/{id}`
   (gives `radiusUsers[0]` + `locationId`) -> `GET .../address-search/v3/addresses/id/{locationId}`
   (gives the actual street address, city, postcode, region). Region name
   is mapped to a 3-char ISO code via `NZ_Regions.xlsx`.

This is a data-prep step that calls out to Voyager (not OneBill) — no
OneBill API calls happen here. Output feeds both `06_Create_Addresses.ipynb`
and `07_Create_Subscription_Orders.ipynb` (the latter now uses the resolved
`radius_user` instead of `SubscriptionLabel`).

> **TODO**: confirm the `Reference` column name
> (`SUBSCRIPTION_REFERENCE_COLUMN` in `onebill_common.py`, currently
> `"Reference"`).


## 1. Setup

In [24]:
import sys, pathlib
sys.path.insert(0, str(pathlib.Path.cwd()))

from onebill_common import *  # noqa: F401,F403
from sqlalchemy import create_engine
from concurrent.futures import ThreadPoolExecutor

logger = get_logger("fetch_subscriptions")

# Use this to limit rows while testing. Set to None once ready for a full run.
TEST_ROW_LIMIT = None
BATCH_NUMBER =  os.environ["BATCH_NUMBER"]

## 2. Pull every subscription from MySQL

In [25]:
assert BI_DATASTORE_URL, "DB_USERNAME/DB_PASSWORD/DB_HOST not set in .env"
engine = create_engine(BI_DATASTORE_URL)

SUBSCRIPTION_QUERY = '''
SELECT * FROM bi_datastore.billing_subscription
WHERE _DataSource = 'vBill'
AND AccountCode = '99965692'
AND SubscriptionEndDate IS NULL
ORDER BY RAND();
'''.strip()

df_subscriptions = pd.read_sql(SUBSCRIPTION_QUERY, con=engine)
logger.info(f"Loaded {len(df_subscriptions):,} subscriptions from MySQL")

if TEST_ROW_LIMIT is not None:
    df_subscriptions = df_subscriptions.head(TEST_ROW_LIMIT)  # Testing limiter — remove/raise for a full run.
    logger.info(f"TEST_ROW_LIMIT active — trimmed to {len(df_subscriptions):,} rows")

df_subscriptions.head()


2026-07-29 06:52:34,197 [INFO] Loaded 264 subscriptions from MySQL


,_rowid,_rowmodified,_sourceid,_DataSource,AccountCode,ServiceType,SubscriptionUSN,SubscriptionLabel,SubscriptionStartDate,SubscriptionEndDate,...,CircuitType,Server,CustomerSuppliedReference,_notforreports_VoyagerOrderHistory,_notforreports_LegacyServiceDescription,NextPlanCode,NextPlanStartDate,NextQuantity,NextCustomPrice,SalesAgentCode
0,4170938565,2026-07-03 20:36:55,260923,vBill,99965692,Broadband - Fibre,V113063044,1.4johnston@williamsinternet.com,2025-12-04,None,...,UFB 100/20/2.5/2.5,None,WC CHCH T9,CPP 3501017,None,None,None,None,None,None
1,4241229553,2026-07-03 20:37:21,262299,vBill,99965692,Broadband - Fibre,V113074082,6.13biddle@williamsinternet.com,2026-02-16,None,...,UFB 100/20/2.5/2.5,None,WC - WC CHCH T9,CPP 3504681,None,None,None,None,None,None
2,4287607035,2026-07-03 20:24:22,263360,vBill,99965692,Broadband - Fibre,V113082523,13.42porutu@williamsinternet.com,2026-04-02,None,...,UFB 100/20/2.5/2.5,None,WC - WC CHCH T9,CPP 3505462,None,None,None,None,None,None
3,4308767691,2026-07-03 20:31:31,263830,vBill,99965692,Broadband - Fibre,V113085997,101.87marine@williamsinternet.com,2026-04-24,None,...,UFB 100/20/2.5/2.5,None,Managed by Williams Limited,CPP 1007059,None,None,None,None,None,None
4,4224347590,2026-07-03 20:37:19,261923,vBill,99965692,Broadband - Fibre,V113071146,406.176manchester@williamsinternet.com,2026-01-29,None,...,UFB 100/20/2.5/2.5,None,Managed by Williams Limited,CPP 1006801,None,None,None,None,None,None


## 3. Resolve target account for every subscription

`TargetAccountNumber` is resolved by `resolve_target_accounts()` (in
`onebill_common.py`, shared with `05_Fetch_Inactive_Subscriptions.ipynb` for
when that comes back into scope), in this priority order:

1. **Managed by Williams** — `Reference` contains the full phrase or the
   `MBW` abbreviation -> the shared `managed_by_williams` bucket account.
   Overrides own-account routing below, even if the subscription's own
   account exists.
2. **Fixed-reference accounts** — `Reference` matches one of
   `FIXED_REFERENCE_ACCOUNTS` (currently: Williams Real Estate, Toa Koura
   Limited, Design by Williams) -> that marker's own fixed, already-existing
   OneBill account. Same priority as step 1 — also overrides own-account
   routing.
3. **Own account** — matched by `AccountCode` against
   `04_Create_Accounts.ipynb`'s real results (`load_account_code_batch_map`).
   This is the actual OneBill `accountNumber` for that subscription's real
   account.
4. **Williams Corporation fallback** — if the subscription's own account has
   no successful (`created`/`exists`) row in `account_results` (not
   migrated, or failed), and it isn't caught by steps 1–2 either.

Run `04_Create_Accounts.ipynb` before this notebook if you haven't already.


In [26]:
own_account_map = load_account_code_batch_map()  # {AccountCode: AccountCode_Batch}, every account in 04's results
real_account_numbers = load_real_target_account_numbers()  # {"managed_by_williams": "...", "williams_corporation": "..."}

missing_keys = set(TARGET_ACCOUNTS) - set(real_account_numbers)
if missing_keys:
    logger.warning(
        f"No successful account_results row found for: {missing_keys} — "
        f"falling back to the TARGET_ACCOUNTS placeholder for those. "
        f"Run 04_Create_Accounts.ipynb (or check it for failures) before trusting this run."
    )

logger.info(f"{len(own_account_map):,} accounts available from 04_Create_Accounts.ipynb (own-account routing)")

reference_col = SUBSCRIPTION_REFERENCE_COLUMN if SUBSCRIPTION_REFERENCE_COLUMN in df_subscriptions.columns else None
if reference_col is None:
    logger.warning(
        f"Column '{SUBSCRIPTION_REFERENCE_COLUMN}' not found in df_subscriptions — "
        f"Managed-by-Williams / fixed-reference routing and the bucket fallback can't work without it. "
        f"Available columns: {list(df_subscriptions.columns)}"
    )
    df_subscriptions["CustomerSuppliedReference"] = None
else:
    df_subscriptions["CustomerSuppliedReference"] = df_subscriptions[reference_col]

df_subscriptions = resolve_target_accounts(df_subscriptions, own_account_map, real_account_numbers)

missing_own_account = (df_subscriptions["TargetAccountKey"] == "williams_corporation")
if missing_own_account.any():
    logger.warning(
        f"{missing_own_account.sum():,} subscriptions have no matching created/existing account in "
        f"04_Create_Accounts.ipynb's results and weren't caught by Managed-by-Williams or a "
        f"fixed-reference account — routed to the Williams Corporation bucket account instead."
    )

logger.info(df_subscriptions["TargetAccountKey"].value_counts(dropna=False).to_string())
df_subscriptions[["SubscriptionUSN", "AccountCode", "CustomerSuppliedReference", "TargetAccountKey", "TargetAccountNumber"]].head(20)


2026-07-29 06:52:34,232 [WARNING] No successful account_results row found for: {'williams_corporation'} — falling back to the TARGET_ACCOUNTS placeholder for those. Run 04_Create_Accounts.ipynb (or check it for failures) before trusting this run.
2026-07-29 06:52:34,235 [INFO] 2 accounts available from 04_Create_Accounts.ipynb (own-account routing)
2026-07-29 06:52:34,245 [WARNING] 114 subscriptions have no matching created/existing account in 04_Create_Accounts.ipynb's results and weren't caught by Managed-by-Williams or a fixed-reference account — routed to the Williams Corporation bucket account instead.
2026-07-29 06:52:34,249 [INFO] TargetAccountKey
managed_by_williams     147
williams_corporation    114
toa_koura                 1
design_by_williams        1
williams_real_estate      1


,SubscriptionUSN,AccountCode,CustomerSuppliedReference,TargetAccountKey,TargetAccountNumber
0,V113063044,99965692,WC CHCH T9,williams_corporation,SR1602
1,V113074082,99965692,WC - WC CHCH T9,williams_corporation,SR1602
2,V113082523,99965692,WC - WC CHCH T9,williams_corporation,SR1602
3,V113085997,99965692,Managed by Williams Limited,managed_by_williams,SR1404
4,V113071146,99965692,Managed by Williams Limited,managed_by_williams,SR1404
5,V113070452,99965692,MBW Christchurch,managed_by_williams,SR1404
6,V113067102,99965692,Managed by Williams Limited,managed_by_williams,SR1404
7,V113063150,99965692,WC CHCH T9,williams_corporation,SR1602
8,V113072458,99965692,Toa Koura Limited,toa_koura,ACCT2303
9,V113079602,99965692,Managed by Williams Limited,managed_by_williams,SR1404


## 4. Look up each subscription's real address + radius username (Voyager)

Two API calls per subscription, keyed on `SupplierServiceID`:
`fetch_voyager_circuit` -> `fetch_voyager_address`, wrapped by
`get_voyager_address`. Run in parallel (it's now real network calls, not
pure string parsing) via the same `ThreadPoolExecutor` pattern used in
`06_Create_Addresses.ipynb`.

Only ACTIVE subscriptions reach this point — inactive ones were split off
in step 2b and don't go through Voyager at all.

Subscriptions with no `SupplierServiceID`, or where either Voyager call
fails, come back with `parsed_ok = False` — flagged the same way unparsed
labels used to be, and skipped by `06_Create_Addresses.ipynb`.


In [27]:
if VOYAGER_CCP_KEY is None or VOYAGER_PARTNER_ID is None:
    logger.warning("VOYAGER_CCP_KEY / VOYAGER_PARTNER_ID not set — every Voyager lookup below will fail. Set them in .env.")

blank_supplier_ids = df_subscriptions["SupplierServiceID"].isna() | (df_subscriptions["SupplierServiceID"].astype(str).str.strip() == "")
if blank_supplier_ids.all():
    logger.warning(
        "SupplierServiceID is blank for EVERY subscription in this batch — the Voyager circuits lookup "
        "can't run at all without it, so every ParsedAddress_* field below will be None. Check the MySQL "
        "source data / SUBSCRIPTION_QUERY in step 2 before re-running."
    )
elif blank_supplier_ids.any():
    logger.warning(f"{blank_supplier_ids.sum():,} / {len(df_subscriptions):,} subscriptions have a blank SupplierServiceID")

voyager_session = new_voyager_session(max_workers=MAX_WORKERS)


def _lookup_row(supplier_service_id):
    return get_voyager_address(voyager_session, supplier_service_id)


logger.info(f"Looking up Voyager address for {len(df_subscriptions):,} subscriptions with {MAX_WORKERS} workers...")

with ThreadPoolExecutor(max_workers=MAX_WORKERS) as executor:
    voyager_results = list(executor.map(_lookup_row, df_subscriptions["SupplierServiceID"]))

existing_parsed_cols = [c for c in df_subscriptions.columns if c.startswith("ParsedAddress_")]
if existing_parsed_cols:
    df_subscriptions = df_subscriptions.drop(columns=existing_parsed_cols)  # safe to re-run this cell

address_parts = pd.DataFrame(voyager_results, index=df_subscriptions.index)
address_parts = address_parts.add_prefix("ParsedAddress_")
df_subscriptions = pd.concat([df_subscriptions, address_parts], axis=1)

unparsed = df_subscriptions[~df_subscriptions["ParsedAddress_parsed_ok"]]
if not unparsed.empty:
    logger.warning(f"{len(unparsed):,} subscriptions could not be resolved to a Voyager address")
    error_summary = (
        unparsed["ParsedAddress_error"].value_counts(dropna=False)
        .rename_axis("error").reset_index(name="count")
    )
    logger.info("Error breakdown:\n" + error_summary.to_string(index=False))

df_subscriptions[[
    "SubscriptionLabel", "SupplierServiceID",
    "ParsedAddress_addLine1", "ParsedAddress_addLine2", "ParsedAddress_city",
    "ParsedAddress_postcode", "ParsedAddress_region_iso", "ParsedAddress_region_code_raw", "ParsedAddress_radius_user",
    "ParsedAddress_parsed_ok", "ParsedAddress_error",
]].head(20)


2026-07-29 06:54:09,487 [WARNING] 3 / 264 subscriptions have a blank SupplierServiceID
2026-07-29 06:54:09,495 [INFO] Looking up Voyager address for 264 subscriptions with 3 workers...
2026-07-29 06:54:42,836 [WARNING] Voyager 500 on https://api.voyager.nz/fibre/v1/circuits/1643316195 — retry 1/3 in 2.3s
2026-07-29 06:54:43,320 [WARNING] Voyager 500 on https://api.voyager.nz/fibre/v1/circuits/ENVOYB02643181 — retry 1/3 in 2.3s
2026-07-29 06:54:43,762 [WARNING] Voyager 429 on https://api.voyager.nz/address-search/v3/addresses/id/3b66fc53d360061d9a48c74184ef9d1100b388c9 — retry 1/6 in 30.5s
2026-07-29 06:54:45,629 [WARNING] Voyager 500 on https://api.voyager.nz/fibre/v1/circuits/1643316195 — retry 2/3 in 4.4s
2026-07-29 06:54:46,132 [WARNING] Voyager 500 on https://api.voyager.nz/fibre/v1/circuits/ENVOYB02643181 — retry 2/3 in 4.4s
2026-07-29 06:54:50,611 [WARNING] Voyager 500 on https://api.voyager.nz/fibre/v1/circuits/1643316195 — retry 3/3 in 8.3s
2026-07-29 06:54:51,123 [WARNING] Voy

,SubscriptionLabel,SupplierServiceID,ParsedAddress_addLine1,ParsedAddress_addLine2,ParsedAddress_city,ParsedAddress_postcode,ParsedAddress_region_iso,ParsedAddress_region_code_raw,ParsedAddress_radius_user,ParsedAddress_parsed_ok,ParsedAddress_error
0,1.4johnston@williamsinternet.com,1643175711,1/4 JOHNSTON GROVE,TAITA,LOWER HUTT,5011,WGN,WGN,1.4johnston@williamsinternet.com,True,None
1,6.13biddle@williamsinternet.com,1643250002,6/13 BIDDLE CRESCENT,TAITA,LOWER HUTT,5011,WGN,WGN,6.13biddle@williamsinternet.com,True,None
2,13.42porutu@williamsinternet.com,1643306233,13/43 PORUTU STREET,FAIRFIELD,LOWER HUTT,5011,WGN,WGN,13.43porutu@williamsinternet.com,True,None
3,101.87marine@williamsinternet.com,ENVOYB02643452,101/87 MARINE PARADE,NORTH NEW BRIGHTON,CHRISTCHURCH,8083,CAN,CAN,101.87marine@williamsinternet.com,True,None
4,406.176manchester@williamsinternet.com,ENVOYB02627012,406/176 MANCHESTER STREET,CHRISTCHURCH CENTRAL,CHRISTCHURCH,8011,CAN,CAN,406.176manchester@williamsinternet.com,True,None
5,166a.manchester@williamsinternet.com,ENVOYB02626432,166A MANCHESTER STREET,CHRISTCHURCH CENTRAL,CHRISTCHURCH,8011,CAN,CAN,166a.manchester@williamsinternet.com,True,None
6,502.162manchesterst@williamsinternet.com,ENVOYB02622952,502/162 MANCHESTER STREET,CHRISTCHURCH CENTRAL,CHRISTCHURCH,8011,CAN,CAN,502.162manchesterst@williamsinternet.com,True,None
7,2.15biddle@williamsinternet.com,1643175882,2/15 BIDDLE CRESCENT,TAITA,LOWER HUTT,5011,WGN,WGN,2.15biddle@williamsinternet.com,True,None
8,165.england@williamsinternet.com,ENVOYB02628428,165 ENGLAND STREET,LINWOOD,CHRISTCHURCH,8011,CAN,CAN,165.england@williamsinternet.com,True,None
9,113.20bath@williamsinternet.com,ENVOYB02636040,113/20 BATH STREET,CHRISTCHURCH CENTRAL,CHRISTCHURCH,8011,CAN,CAN,113.20bath@williamsinternet.com,True,None


In [28]:
# Second pass: retry subscriptions that failed with a transient 5xx on the
# first pass, after a longer cooldown. 404s / "no SupplierServiceID" are
# permanent and intentionally skipped -- see voyager_second_pass() docstring.
SECOND_PASS_DELAY_SECONDS = 90  # bump this up if the same circuits still fail after 90s

df_subscriptions = voyager_second_pass(
    df_subscriptions,
    voyager_session,
    delay_seconds=SECOND_PASS_DELAY_SECONDS,
    max_workers=MAX_WORKERS,
)

unparsed = df_subscriptions[~df_subscriptions["ParsedAddress_parsed_ok"]]
if not unparsed.empty:
    error_summary = (
        unparsed["ParsedAddress_error"].value_counts(dropna=False)
        .rename_axis("error").reset_index(name="count")
    )
    logger.info(f"{len(unparsed):,} subscriptions still unresolved overall:\n" + error_summary.to_string(index=False))
else:
    logger.info("All subscriptions resolved.")

unparsed[["SubscriptionUSN", "SupplierServiceID", "ParsedAddress_error"]]


2026-07-29 07:02:32,441 [INFO] Voyager second pass: 14 subscriptions failed with a transient 5xx on the first pass — waiting 90s before retrying...
2026-07-29 07:04:16,471 [INFO] Voyager second pass: 14 / 14 resolved on retry.
2026-07-29 07:04:16,475 [INFO] 8 subscriptions still unresolved after second pass:
                                                                                                               error  count
                                                                           no SupplierServiceID on this subscription      3
circuits lookup failed: 404 Client Error: Not Found for url: https://api.voyager.nz/fibre/v1/circuits/ENVOYB02624632      1
circuits lookup failed: 404 Client Error: Not Found for url: https://api.voyager.nz/fibre/v1/circuits/ENVOYB02622491      1
    circuits lookup failed: 404 Client Error: Not Found for url: https://api.voyager.nz/fibre/v1/circuits/1643259206      1
circuits lookup failed: 404 Client Error: Not Found for url: https://a

,SubscriptionUSN,SupplierServiceID,ParsedAddress_error
59,V113068670,ENVOYB02624632,circuits lookup failed: 404 Client Error: Not ...
106,V113062335,None,no SupplierServiceID on this subscription
108,V113066765,ENVOYB02622491,circuits lookup failed: 404 Client Error: Not ...
143,V113074579,1643259206,circuits lookup failed: 404 Client Error: Not ...
163,V113071120,ENVOYB02627005,circuits lookup failed: 404 Client Error: Not ...
198,V113076806,ENVOYB02633693,circuits lookup failed: 404 Client Error: Not ...
206,V113071849,None,no SupplierServiceID on this subscription
242,V113062343,None,no SupplierServiceID on this subscription


In [29]:
df_subscriptions["SupplierServiceID"] = df_subscriptions["SupplierServiceID"] + BATCH_NUMBER
df_subscriptions["ParsedAddress_radius_user"] = df_subscriptions["ParsedAddress_radius_user"] + BATCH_NUMBER
df_subscriptions["SubscriptionUSN"] = df_subscriptions["SubscriptionUSN"] + BATCH_NUMBER
df_subscriptions.head()

,_rowid,_rowmodified,_sourceid,_DataSource,AccountCode,ServiceType,SubscriptionUSN,SubscriptionLabel,SubscriptionStartDate,SubscriptionEndDate,...,ParsedAddress_addLine2,ParsedAddress_city,ParsedAddress_postcode,ParsedAddress_region_name,ParsedAddress_region_iso,ParsedAddress_region_code_raw,ParsedAddress_radius_user,ParsedAddress_location_id,ParsedAddress_parsed_ok,ParsedAddress_error
0,4170938565,2026-07-03 20:36:55,260923,vBill,99965692,Broadband - Fibre,V113063044,1.4johnston@williamsinternet.com,2025-12-04,None,...,TAITA,LOWER HUTT,5011,WELLINGTON REGION,WGN,WGN,1.4johnston@williamsinternet.com,3954ff598cd75efce40f2e3e8494c21d38222db9,True,None
1,4241229553,2026-07-03 20:37:21,262299,vBill,99965692,Broadband - Fibre,V113074082,6.13biddle@williamsinternet.com,2026-02-16,None,...,TAITA,LOWER HUTT,5011,WELLINGTON REGION,WGN,WGN,6.13biddle@williamsinternet.com,adc21fb56cd10016ad4afbf79146e3443bd4f447,True,None
2,4287607035,2026-07-03 20:24:22,263360,vBill,99965692,Broadband - Fibre,V113082523,13.42porutu@williamsinternet.com,2026-04-02,None,...,FAIRFIELD,LOWER HUTT,5011,WELLINGTON REGION,WGN,WGN,13.43porutu@williamsinternet.com,bddd2163c6fc1e401b2ebc8dfe095441c6c61e26,True,None
3,4308767691,2026-07-03 20:31:31,263830,vBill,99965692,Broadband - Fibre,V113085997,101.87marine@williamsinternet.com,2026-04-24,None,...,NORTH NEW BRIGHTON,CHRISTCHURCH,8083,CANTERBURY REGION,CAN,CAN,101.87marine@williamsinternet.com,d8b76bd532e8c6f315df61923a2cad6ed1508869,True,None
4,4224347590,2026-07-03 20:37:19,261923,vBill,99965692,Broadband - Fibre,V113071146,406.176manchester@williamsinternet.com,2026-01-29,None,...,CHRISTCHURCH CENTRAL,CHRISTCHURCH,8011,CANTERBURY REGION,CAN,CAN,406.176manchester@williamsinternet.com,503965ce62681f76ddf38aba7f35d577ace56fcc,True,None


## 5. Save

In [30]:
save_df("subscriptions_resolved", df_subscriptions)


Saved 264 rows -> migration_data\05_subscriptions_resolved.csv
